# EuroSAT Land Classifier → Deforestation Detector
**Step 1 of the project: train the classifier.**

Run each cell from top to bottom (Shift+Enter). If anything errors, copy the full error message and send it back — don't try to fix it yourself, just paste it.


## 0. Turn on the free GPU
Before running anything: click **Runtime** (top menu) → **Change runtime type** → select **T4 GPU** → **Save**. Then run the cell below to confirm it worked.

In [ ]:
import torch
print("GPU available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None - go back and turn on the T4 GPU above")


## 1. Install and load the dataset
We're using EuroSAT hosted on Hugging Face — no account or download needed.

In [ ]:
!pip install -q datasets

from datasets import load_dataset

ds = load_dataset("blanchon/EuroSAT_RGB")
print(ds)
print("Columns:", ds['train'].column_names)
print("Classes:", ds['train'].features['label'].names)
print("Total images:", len(ds['train']))


## 2. Look at a few sample images
Just a sanity check that the data loaded correctly.

In [ ]:
import matplotlib.pyplot as plt

class_names = ds['train'].features['label'].names
fig, axes = plt.subplots(1, 5, figsize=(15, 3))
for i, ax in enumerate(axes):
    item = ds['train'][i * 2000]
    ax.imshow(item['image'])
    ax.set_title(class_names[item['label']])
    ax.axis('off')
plt.show()


## 3. Split into train / validation / test
80% train, 10% validation, 10% test.

In [ ]:
split1 = ds['train'].train_test_split(test_size=0.2, seed=42)
train_data = split1['train']
split2 = split1['test'].train_test_split(test_size=0.5, seed=42)
val_data = split2['train']
test_data = split2['test']

print("Train:", len(train_data), "Val:", len(val_data), "Test:", len(test_data))


## 4. Define preprocessing (resize, normalize, augment)

In [ ]:
from torchvision import transforms

mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])


## 5. Wrap the dataset for PyTorch

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class EuroSATWrapper(Dataset):
    def __init__(self, hf_dataset, transform):
        self.dataset = hf_dataset
        self.transform = transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        item = self.dataset[idx]
        image = item['image'].convert('RGB')
        label = item['label']
        image = self.transform(image)
        return image, label

train_set = EuroSATWrapper(train_data, train_transform)
val_set = EuroSATWrapper(val_data, eval_transform)
test_set = EuroSATWrapper(test_data, eval_transform)

train_loader = DataLoader(train_set, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_set, batch_size=32, shuffle=False, num_workers=2)
test_loader = DataLoader(test_set, batch_size=32, shuffle=False, num_workers=2)


## 6. Build the model (ResNet50, transfer learning)

In [ ]:
import torchvision.models as models
import torch.nn as nn

model = models.resnet50(weights='IMAGENET1K_V2')

for param in model.parameters():
    param.requires_grad = False

num_features = model.fc.in_features
model.fc = nn.Linear(num_features, len(class_names))
model = model.to('cuda')
print("Model ready. Output classes:", len(class_names))


## 7. Train the final layer (head only)
This should take a few minutes on the T4 GPU.

In [ ]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

num_epochs = 10
history = {'loss': [], 'val_acc': []}

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to('cuda'), labels.to('cuda')
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to('cuda'), labels.to('cuda')
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    val_acc = correct / total
    avg_loss = running_loss / len(train_loader)
    history['loss'].append(avg_loss)
    history['val_acc'].append(val_acc)
    print(f"Epoch {epoch+1}/{num_epochs} - Loss: {avg_loss:.4f} - Val Accuracy: {val_acc:.4f}")


## 8. Fine-tune the whole model at a lower learning rate
This squeezes out extra accuracy. Should push you toward 95%+.

In [ ]:
for param in model.parameters():
    param.requires_grad = True

optimizer = optim.Adam(model.parameters(), lr=0.0001)
num_epochs_finetune = 8

for epoch in range(num_epochs_finetune):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to('cuda'), labels.to('cuda')
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to('cuda'), labels.to('cuda')
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    val_acc = correct / total
    avg_loss = running_loss / len(train_loader)
    history['loss'].append(avg_loss)
    history['val_acc'].append(val_acc)
    print(f"Fine-tune Epoch {epoch+1}/{num_epochs_finetune} - Loss: {avg_loss:.4f} - Val Accuracy: {val_acc:.4f}")


## 9. Plot training curves

In [ ]:
plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plt.plot(history['loss'])
plt.title('Training Loss')
plt.xlabel('Epoch')

plt.subplot(1,2,2)
plt.plot(history['val_acc'])
plt.title('Validation Accuracy')
plt.xlabel('Epoch')
plt.tight_layout()
plt.savefig('training_curves.png')
plt.show()


## 10. Final evaluation on the held-out test set
This is the number that goes in your writeup.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to('cuda')
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())

print(classification_report(all_labels, all_preds, target_names=class_names))


## 11. Confusion matrix

In [ ]:
import seaborn as sns

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=class_names, yticklabels=class_names, cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix - EuroSAT Test Set')
plt.tight_layout()
plt.savefig('confusion_matrix.png')
plt.show()


## 12. Save the trained model
This saves the weights to your Colab session. Run the next cell too, to download it to your computer so it isn't lost when the session ends.

In [ ]:
torch.save(model.state_dict(), 'eurosat_resnet50.pth')
print("Saved.")


In [ ]:
from google.colab import files
files.download('eurosat_resnet50.pth')
files.download('confusion_matrix.png')
files.download('training_curves.png')


## Done with Step 1
Send me back:
1. The final test accuracy number (from the classification_report output)
2. The confusion matrix image
3. Any class that performed noticeably worse than the others

Then we move to **Step 2: applying this model to real Borneo satellite imagery to detect deforestation.**
